In [27]:
import calendar
import json
import os
import pickle
import random
import re
import sys
from datetime import date
from typing import List

import dateutil
import graphistry
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
import seaborn as sns
from pyspark.sql import DataFrame, SparkSession

In [2]:
GRAPHISTRY_KEY_ID = os.environ["GRAPHISTRY_KEY_ID"]
GRAPHISTRY_API_KEY = os.environ["GRAPHISTRY_API_KEY"]

In [3]:
# Configuration for Graphistry
GRAPHISTRY_PARAMS = {
    "play": 500,
    "pointOpacity": 0.7,
    "edgeOpacity": 0.3,
    "edgeCurvature": 0.3,
    "showArrows": True,
    "gravity": 0.15,
    "showPointsOfInterestLabel": False,
    "labels": {
        "shortenLabels": False,
    },
}
FAVICON_URL = "https://graphlet.ai/assets/icons/favicon.ico"
LOGO = {"url": "https://graphlet.ai/assets/Branding/Graphlet%20AI.svg", "dimensions": {"maxWidth": 100, "maxHeight": 100}}

In [4]:
# Initialize a SparkSession
spark: SparkSession = (
    SparkSession.builder.appName("Stack Overflow Pregel API")
    # Lets the Id:(Stack Overflow int) and id:(GraphFrames ULID) coexist
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints")

25/04/18 12:30:53 WARN Utils: Your hostname, Achilles.local resolves to a loopback address: 127.0.0.1; using 10.0.0.246 instead (on interface en0)
25/04/18 12:30:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 12:30:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Evaluating the Extracted Data

We used BAML and Gemini 2.5. Pro to extract the entities listed below. Let's inspect them for quality problems!

In [5]:
!ls ../data/knowledge_graph

companies.parquet                    products.parquet
company_ticker_relationships.parquet tech_company_relationships.parquet
documents.parquet                    technologies.parquet
product_company_relationships        tickers.parquet


## Evaluating Nodes

Our nodes are `companies`, `products`, `technologies` and `tickers`.

In [17]:
company_df = spark.read.parquet("../data/knowledge_graph/companies.parquet")
print(f"Company Count: {company_df.count():,}")
company_df.show(5)

Company Count: 354
+----+--------------------+---------+------------+---------------------+------------+-----------------+-----------+------+-----------+
| ceo|         description|employees|founded_year|headquarters_location|linkedin_url|             name|revenue_usd|ticker|website_url|
+----+--------------------+---------+------------+---------------------+------------+-----------------+-----------+------+-----------+
|NULL|ACM Research is a...|     NULL|        1998|           California|        NULL|     ACM Research|  250000000|  NULL|       NULL|
|NULL|AMD is a semicond...|     NULL|        NULL|                 NULL|        NULL|              AMD|       NULL|  NULL|       NULL|
|NULL|Shanghai-based co...|     NULL|        NULL|                 NULL|        NULL|             AMEC|       NULL|  NULL|       NULL|
|NULL|Purchased lithogr...|     NULL|        NULL|                 NULL|        NULL|              ASE|       NULL|  NULL|       NULL|
|NULL|A major toolmaker...|     NULL

In [16]:
product_df = spark.read.parquet("../data/knowledge_graph/products.parquet")
print(f"Product Count: {product_df.count():,}")
product_df.show(5)

Product Count: 411
+--------------------+--------------------+--------------------+------------+
|             company|         description|                name|company_name|
+--------------------+--------------------+--------------------+------------+
|{NULL, Taiwan Sem...|A semiconductor m...|         3nm FinFlex|        TSMC|
|{NULL, Qualcomm i...|A PCIe inline acc...|5G Distributed Un...|    Qualcomm|
|{NULL, Qualcomm i...|Entire solutions ...|5G front end modules|    Qualcomm|
|{NULL, The larges...|Capabilities for ...|800G optical tran...|       Cisco|
|{NULL, Nvidia is ...|         Nvidia GPU.|                A100|      Nvidia|
+--------------------+--------------------+--------------------+------------+
only showing top 5 rows



In [14]:
technology_df = spark.read.parquet("../data/knowledge_graph/technologies.parquet")
print(f"Technology Count: {technology_df.count():,}")
technology_df.show(5)

Technology Count: 356
+--------------------+--------------------+-----------+---------------+
|         description|           developer|       name| developer_name|
+--------------------+--------------------+-----------+---------------+
|Process node orig...|{NULL, Intel is t...|       10nm|          Intel|
|The most advanced...|{NULL, foundry, N...|       12nm|GlobalFoundries|
|Process node wher...|{NULL, Intel is t...|       14nm|          Intel|
|Almost all equipm...|{NULL, China’s la...|14nm FinFET|           SMIC|
|Semiconductor man...|{NULL, Taiwan Sem...|       16nm|           TSMC|
+--------------------+--------------------+-----------+---------------+
only showing top 5 rows



In [25]:
ticker_df = spark.read.parquet("../data/knowledge_graph/tickers.parquet")
print(f"Ticker Count: {ticker_df.count():,}")
ticker_df.filter(F.col("name").isNotNull()).orderBy("name").show(10)

Ticker Count: 48


NameError: name 'F' is not defined

In [20]:
doc_df = spark.read.parquet("../data/knowledge_graph/documents.parquet")
print(f"Total Documents: {doc_df.count():,}")
doc_df.show(5)

Total Documents: 171
+--------------------+------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+
|             authors|collected_at|           companies|            products|published_at|             summary|        technologies|             tickers|               title|
+--------------------+------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+
|[{SemiAnalysis, N...|        NULL|[{NULL, Nvidia is...|[{{NULL, Nvidia i...|        NULL|Nvidia was hacked...|[{Technology that...|[{NULL, Nvidia, N...|Nvidia was hacked...|
|                NULL|        NULL|[{NULL, China’s a...|[{{NULL, China’s ...|        NULL|Biren, China’s ar...|                  []|                NULL|The regulations a...|
|                NULL|        NULL|                  []|                  []|        NULL|The article di

## Evaluating Edges

Our edge types are `Company--ListedUnder-->Ticker`, `Company--Sells-->Product` and `Company--Develops-->Technology`.

In [31]:
!ls ../data/knowledge_graph

companies.parquet                     products.parquet
company_ticker_relationships.parquet  tech_company_relationships.parquet
documents.parquet                     technologies.parquet
product_company_relationships.parquet tickers.parquet


## Build Bidirectioal Company / Product Edges

We need bidirectional edges: `Company--Sells-->Product` and `Product--Soldby-->Company`.

In [56]:
product_company_df = spark.read.parquet("../data/knowledge_graph/product_company_relationships.parquet")
print(f"Total Company--Sells-->Product Edges: {product_company_df.count():,}")
(
    product_company_df.select("company_name", "product_name")
    .orderBy("company_name", "product_name")
    .show(36, False)
)

Total Company--Sells-->Product Edges: 411
+--------------------------------+------------------------------------------+
|company_name                    |product_name                              |
+--------------------------------+------------------------------------------+
|A cross-university collaboration|Vicuna                                    |
|ACM Research                    |SAPS single wafer cleaning tool           |
|ACM Research                    |TEBO (Timely Energized Bubble Oscillation)|
|ACM Research                    |electro-chemical plating tools            |
|AMD                             |Bergamo                                   |
|AMD                             |EPYC 9654 Genoa                           |
|AMD                             |Genoa                                     |
|AMD                             |MI300                                     |
|AMD                             |MI300X                                    |
|AMD                  

In [49]:
co_sells_product_df = product_company_df.selectExpr("company_name AS src", "product_name AS dst", "'Sells' AS relationship")
co_sells_product_df.show(10)

+--------------------+--------------------+------------+
|                 src|                 dst|relationship|
+--------------------+--------------------+------------+
|              Nvidia|     Bluefield-3 DPU|       Sells|
|              Nvidia|Base Command Mana...|       Sells|
|               Intel|       Sierra Forest|       Sells|
|              Nvidia|                A100|       Sells|
|Advanced Micro De...|             Navi 32|       Sells|
|                Meta|           Quest Pro|       Sells|
|              Google|          TPUv2 pods|       Sells|
|               Nuvia|  Nuvia Phoenix core|       Sells|
|              OpenAI|            DALL-E 3|       Sells|
|             Marvell|    LiquidSecurity 2|       Sells|
+--------------------+--------------------+------------+
only showing top 10 rows



In [51]:
prod_sold_by_co_df = product_company_df.selectExpr("product_name AS src", "company_name AS dst", "'SoldBy' AS relationship")
prod_sold_by_co_df.show(10)

+--------------------+--------------------+------------+
|                 src|                 dst|relationship|
+--------------------+--------------------+------------+
|     Bluefield-3 DPU|              Nvidia|      SoldBy|
|Base Command Mana...|              Nvidia|      SoldBy|
|       Sierra Forest|               Intel|      SoldBy|
|                A100|              Nvidia|      SoldBy|
|             Navi 32|Advanced Micro De...|      SoldBy|
|           Quest Pro|                Meta|      SoldBy|
|          TPUv2 pods|              Google|      SoldBy|
|  Nuvia Phoenix core|               Nuvia|      SoldBy|
|            DALL-E 3|              OpenAI|      SoldBy|
|    LiquidSecurity 2|             Marvell|      SoldBy|
+--------------------+--------------------+------------+
only showing top 10 rows



## Build Bidirectional Company / Ticker Symbol Edges

We need bidirectional edges: `Company--ListedUnder-->Ticker` and `Ticker--Represents-->Company`.

In [52]:
company_ticker_df = spark.read.parquet("../data/knowledge_graph/company_ticker_relationships.parquet")
print(f"Total Company--ListedUnder-->Ticker Edges: {company_ticker_df.count():,}")
company_ticker_df.show(10)

Total Company--ListedUnder-->Ticker Edges: 23
+--------------------+-------------+
|        company_name|ticker_symbol|
+--------------------+-------------+
|Advanced Micro De...|          AMD|
|Advanced Micro De...|          AMD|
|           Advantest|         6857|
|   Aehr Test Systems|         AEHR|
|      Alphawave Semi|          AWE|
|    Amazon.com, Inc.|         AMZN|
|Amkor Technology,...|         AMKR|
|   Applied Materials|         AMAT|
|               Credo|         CRDO|
|   DISCO Corporation|         6146|
+--------------------+-------------+
only showing top 10 rows



In [ ]:
co_listed_ticker_df = company_ticker_df.selectExpr("product_name AS src", "company_name AS dst", "'SoldBy' AS relationship")
prod_sold_by_co_df.show(10)

## Build Bidirectional Company / Technology Edges

We need bidirectional edges: `Company--Develops-->Technology` and `Technology--DevelopedBy-->Company`.

In [57]:
tech_company_df = spark.read.parquet("../data/knowledge_graph/tech_company_relationships.parquet")
print(f"Total Company--Develops-->Technology Edges: {tech_company_df.count():,}")
(
    tech_company_df.select("company_name", "technology_name")
    .orderBy("company_name", "technology_name")
    .show(36, False)
)

Total Company--Develops-->Technology Edges: 356
+----------------------------+-------------------------------------------+
|company_name                |technology_name                            |
+----------------------------+-------------------------------------------+
|ACM Research                |ALD                                        |
|ACM Research                |CVD                                        |
|ACM Research                |photoresist stripping                      |
|AMD                         |Infinity Cache                             |
|AMD                         |Infinity Fabric                            |
|AMD                         |RCCL                                       |
|AMD                         |ROCm                                       |
|AMD                         |xGMI                                       |
|ASML                        |ArF                                        |
|ASML                        |ArFi                  

## Merge all Edges and Save

We need a single Parquet file for post-processing.

In [54]:
edge_df = co_sells_product_df.union(prod_sold_by_co_df)
print(f"Total Edges: {edge_df.count():,}")
edge_df.sample(0.01).show()

Total Edges: 822
+--------------------+--------------------+------------+
|                 src|                 dst|relationship|
+--------------------+--------------------+------------+
|Advanced Micro De...|              RDNA 3|       Sells|
|                 AWS| AWS ParallelCluster|       Sells|
|              Nvidia|Datacenter and AI...|       Sells|
|Advanced Micro De...|               Genoa|       Sells|
|              Nvidia|                  L2|       Sells|
|Advanced Micro De...|              MI250X|       Sells|
|     Bluefield-3 DPU|              Nvidia|      SoldBy|
|   TPUv5 (Viperfish)|              Google|      SoldBy|
|         Bluefield-3|              Nvidia|      SoldBy|
|             Icelake|   Intel Corporation|      SoldBy|
|                B100|              Nvidia|      SoldBy|
|             KubeCon|           CoreWeave|      SoldBy|
|Maia 100 AI accel...|           Microsoft|      SoldBy|
+--------------------+--------------------+------------+

